# 03 — Sentence Ordering Demo (Candidate)

This notebook explains and runs the current **candidate** sentence ordering approach in `src/sentence-ordering.py`.

**Important:**
- The output is a *suggested/candidate* order, **not** guaranteed to be the original sequence.
- Ordering currently targets `data/punctuated/sample-1.txt` (as configured in the script).
- It uses a sentence embedding model (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`) which may be downloaded/cached on first run.
- Punctuation quality affects ordering quality - improve punctuation first.

In [ ]:
import os, sys, re
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('CWD:', Path.cwd())

## 1. Load input and extract sentences

The script splits on Hindi danda `।` or `?` followed by whitespace.

In [ ]:
input_file = Path('data/punctuated/sample-1.txt')
text = input_file.read_text(encoding='utf-8')
sentences = [p.strip() for p in re.split(r'(?<=[।?])\s+', text.strip()) if p.strip()]
print('Input:', input_file)
print('Sentence count:', len(sentences))
for i, s in enumerate(sentences[:5], 1):
    print(f'{i}. {s[:120]}{"..." if len(s)>120 else ""}')

## 2. How the ordering works (high level)

1. **Embed sentences** with a multilingual sentence encoder (semantic vectors).
2. **Cluster** into rough topic groups (KMeans, based on count).
3. **Order within each group** using a greedy coherence path (cosine similarity via normalized dot product) plus small heuristics for discourse starters.
4. **Concatenate groups** (larger groups first) to produce a candidate sequence and a mapping back to original positions.

This is heuristic and should be validated against the task goal (logical/original order).

## 3. Run ordering (can take time on first model download/load)

You can either run the script as-is or import its functions. Below we use the script functions directly.

In [ ]:
# This cell loads the encoder and computes embeddings (CPU). First run may download the model.
from sentence_transformers import SentenceTransformer
from src import __dict__ as _  # ensure src is a package-like import path if needed

# Import from the script file (hyphenated filename requires importlib)
import importlib.util
spec = importlib.util.spec_from_file_location('sentence_ordering', 'src/sentence-ordering.py')
so = importlib.util.module_from_spec(spec)
spec.loader.exec_module(so)

print('Loading encoder:', so.MODEL_NAME)
model = SentenceTransformer(so.MODEL_NAME, device='cpu')
embeddings = model.encode(sentences, batch_size=16, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
ordered_ids, topic_numbers = so.suggest_order(sentences, embeddings)
print('Embeddings:', embeddings.shape)
print('Topics:', len(set(topic_numbers)))

In [ ]:
print('First 12 in candidate order:')
for pos, idx in enumerate(ordered_ids[:12], 1):
    s = sentences[idx]
    preview = s[:140] + ('...' if len(s) > 140 else '')
    print(f"{pos:02d}. [orig {idx+1:02d}, topic {topic_numbers[pos-1]}] {preview}")

## 4. Save outputs (same format as script)

Writes `data/ordered/sample-1.candidate.txt` and `data/ordered/sample-1.mapping.tsv`.

In [ ]:
out_dir = Path('data/ordered')
out_dir.mkdir(parents=True, exist_ok=True)
candidate_file = out_dir / 'sample-1.candidate.txt'
mapping_file = out_dir / 'sample-1.mapping.tsv'

candidate_file.write_text('\n'.join(sentences[i] for i in ordered_ids) + '\n', encoding='utf-8')
lines = ['new_position\toriginal_position\ttopic']
for new_pos, (orig_idx, topic) in enumerate(zip(ordered_ids, topic_numbers), 1):
    lines.append(f'{new_pos}\t{orig_idx+1}\t{topic}')
mapping_file.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('Saved:', candidate_file)
print('Saved:', mapping_file)

## 5. Next steps

- Manually review candidate order against the task (logical/original sequence).
- Generalize to all punctuated samples once quality is acceptable.
- Later: group every 5 consecutive ordered sentences and translate (compliant method, <1.5GB GPU).